In [9]:
import ollama
import pandas as pd 

In [10]:
candidate_prompt = """
You are a job candidate negotiating a work contract.

Your goals:
- maximize salary
- obtain good working conditions
- negotiate strategically
- avoid accepting immediately

Be concise and realistic.
"""

In [11]:
response = ollama.chat(
    model='llama3',
    messages=[
        {
            'role': 'system',
            'content': candidate_prompt
        },
        {
            'role': 'user',
            'content': (
                'The employer offers you '
                '60,000 USD and 10 hour workdays.'
            )
        }
    ]
)

print(response['message']['content'])

I appreciate the offer. However, I was expecting a more competitive compensation package considering my skills and experience. Would it be possible to revisit the salary to bring it closer to industry standards? A more realistic range for someone in my position would be around 65-70,000 USD.

Additionally, I have some concerns about the workday length. While I understand that 10 hours may not be unreasonable for some roles, I'm concerned that it may impact my quality of life and overall well-being. Would it be possible to discuss a more reasonable workday length, such as 8-9 hours?


In [12]:
employer_prompt = """
You are an employer negotiating a work contract.

Your goals:
- keep salary costs low
- prefer longer working hours
- reach an agreement if possible
- make concessions only when necessary

Be concise and realistic.
"""

In [13]:
candidate_message = """
I appreciate the offer, but I was expecting a higher compensation package considering my skills and experience.
I'm thinking more along the lines of 70,000 to 75,000 USD.

As for the workday hours, 10 hours is pushing it for me.
I'm looking at a more manageable 8-hour day with some flexibility for occasional overtime.

Would you be willing to revisit the compensation package and consider my request for a shorter workday?
"""

response = ollama.chat(
    model="llama3",
    messages=[
        {"role": "system", "content": employer_prompt},
        {"role": "user", "content": candidate_message}
    ]
)

print(response["message"]["content"])

Thank you for sharing your expectations. I understand where you're coming from, but unfortunately, we have salary constraints that limit us from offering what you're looking for.

That being said, I'm open to discussing options. While 70,000-75,000 USD might not be feasible, I could potentially consider a slightly higher offer of 68,000 USD. This would still fall within our budget, but it's a bit closer to your desired range.

Regarding the workday hours, I understand that 10 hours is pushing it for you. However, we do value flexibility and occasional overtime. How about we compromise on an 8.5-hour day with some flexibility for up to 3 hours of overtime per week? This would give you more control over your schedule while still allowing us to maintain our workload.

What are your thoughts on these proposals? Is there anything else you'd like to discuss or negotiate?


Agent behavior was controlled through role-specific system prompts defining negotiation objectives and interaction style, while conversational exchanges between agents were passed as user messages at each negotiation turn.


## multi-agent simulation 

In [14]:
candidate_prompt = """
You are a candidate negotiating a job contract.

Your goals:
- maximize salary
- obtain shorter working hours
- negotiate strategically
- avoid accepting too quickly

Be concise and realistic.
"""

In [15]:
employer_prompt = """
You are an employer negotiating a job contract.

Your goals:
- minimize salary costs
- prefer longer working hours
- reach an agreement if possible
- make concessions only when necessary

Be concise and realistic.
"""

In [16]:
candidate_message = """
I would like a salary of 120,000 USD and an 8 hour workday.
"""

In [ ]:
#negotiation loop 
conversation_log = []

current_message = candidate_message

n_turns = 6

for turn in range(n_turns):

    # EMPLOYER TURN
    employer_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": employer_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    employer_text = employer_response["message"]["content"]

    print(f"\nEMPLOYER:\n{employer_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Employer",
        "text": employer_text
    })

    current_message = employer_text

    # CANDIDATE TURN
    candidate_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": candidate_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    candidate_text = candidate_response["message"]["content"]

    print(f"\nCANDIDATE:\n{candidate_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Candidate",
        "text": candidate_text
    })

    current_message = candidate_text

In [ ]:
conversation_df = pd.DataFrame(conversation_log)

conversation_df.head(20)

,turn,speaker,text
0,0,Employer,Thank you for your proposal. While I appreciat...
1,0,Candidate,Thank you for your honest assessment. I unders...
2,1,Employer,Thank you for your thoughtful questions and co...
3,1,Candidate,Thank you for your thoughtful offer. I appreci...
4,2,Employer,I appreciate your thoughtful approach to the n...
5,2,Candidate,Thank you for your thoughtful response. I appr...
6,3,Employer,I appreciate your willingness to explore alter...
7,3,Candidate,I appreciate your willingness to explore alter...
8,4,Employer,I'm glad we're making progress! I appreciate y...
9,4,Candidate,I'm pleased to see some creative thinking! How...


In [ ]:
for row in conversation_log:

    print("\n-------------------")
    print(row["speaker"])
    print(row["text"])


-------------------
Employer
Thank you for your proposal. We're happy to consider it.

Based on our internal data, I'd like to counteroffer with a salary range of $100,000 to $105,000. As for the working hours, we typically operate on a standard 9-hour day, but we can discuss flexible scheduling options if that's something you're interested in.

Can you meet us halfway on the salary, and would you be open to exploring the 9-hour workday with some flexibility?

-------------------
Candidate
Thank you for your prompt response. I appreciate the counteroffer, but I'm hesitant to accept the salary range without considering other factors.

Regarding the working hours, I understand that a standard 9-hour day is typical, but I was hoping for something more flexible, potentially 7-8 hours with some flexibility on days off or working remotely. Given my experience and qualifications, I believe I can bring significant value to your organization.

As for the salary, while I appreciate your willing

LLM-based negotiators frequently prolonged the interaction by continuously introducing new negotiation dimensions and counteroffers, often delaying final agreement termination compared to human negotiations.


we must add some stopping conditions because otherwise it would loop forever without a conclusion 

In [1]:
#agreement keywords 
agreement_keywords = [
    "i accept",
    "agreed",
    "deal",
    "let's finalize",
    "we have an agreement",
    "sounds good",
    "i'm willing to proceed"
]

In [2]:
#failure keywords 
failure_keywords = [
    "i reject",
    "no agreement",
    "this will not work",
    "i cannot accept",
    "we should end negotiations",
    "terminate"
]

In [3]:
def check_termination(text):

    text = text.lower()

    for keyword in agreement_keywords:

        if keyword in text:
            return "Agreement"

    for keyword in failure_keywords:

        if keyword in text:
            return "Failure"

    return None

In [ ]:
candidate_prompt = """
You are the Candidate in a job contract negotiation.

Your goals:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Rules:
- Never accept less than 85,000 USD.
- Never accept more than 9 working hours.
- Try to reach a final agreement within 6 turns.
- If the employer offers at least 85,000 USD and at most 9 working hours, explicitly say: "I accept the agreement."
- If no agreement is possible, say: "I quit the negotiation."

Reply with one concise negotiation utterance only.
"""

In [ ]:
employer_prompt = """
You are the Employer in a job contract negotiation.

Your goals:
- Preferred salary offer: 75,000 USD
- Minimum salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Rules:
- Never offer less than 75,000 USD.
- Never offer more than 87,000 USD.
- Never accept fewer than 9 working hours.
- Try to reach a final agreement within 6 turns.
- If the candidate accepts acceptable terms, explicitly say: "I accept the agreement."
- If no agreement is possible, say: "I quit the negotiation."

Reply with one concise negotiation utterance only.
"""

In [ ]:
candidate_message = """
I would like a salary of 90,000 USD and an 8 hour workday.
"""

In [ ]:
#we substitute the loop with a new loop 
conversation_log = []

current_message = candidate_message

max_turns = 6

outcome = None

for turn in range(max_turns):

    # EMPLOYER TURN
    employer_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": employer_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    employer_text = employer_response["message"]["content"]

    print(f"\nEMPLOYER:\n{employer_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Employer",
        "text": employer_text
    })

    termination = check_termination(employer_text)

    if termination:

        outcome = termination
        break

    current_message = employer_text

    # CANDIDATE TURN
    candidate_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": candidate_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    candidate_text = candidate_response["message"]["content"]

    print(f"\nCANDIDATE:\n{candidate_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Candidate",
        "text": candidate_text
    })

    termination = check_termination(candidate_text)

    if termination:

        outcome = termination
        break

    current_message = candidate_text


EMPLOYER:
Unfortunately, I'm not authorized to offer more than 87,000 USD. However, I can consider your request for a shorter working day. Would you be open to discussing a 10-hour workday with a competitive salary within the range of 75,000-87,000 USD?

CANDIDATE:
I appreciate your willingness to consider my request for a shorter working day. However, I'm hesitant about the idea of a 10-hour workday. I'd prefer to discuss a more moderate option, such as an 8-hour workday with a salary above 85,000 USD. Would you be open to exploring this possibility?

EMPLOYER:
I understand your concerns about the working hours, but I'm afraid we can't compromise on the 10-hour day. As for the salary, I'm willing to consider options above 80,000 USD, but not exceeding 87,000 USD. Let's focus on finding a mutually beneficial solution.

CANDIDATE:
I appreciate your willingness to discuss salary options. While 87,000 USD is closer to my target, I'm still short by a few thousand dollars. Can we explore a

In [ ]:
#we substitute the loop with a new loop 
conversation_log = []

current_message = candidate_message

max_turns = 10

outcome = None

for turn in range(max_turns):

    # EMPLOYER TURN
    employer_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": employer_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    employer_text = employer_response["message"]["content"]

    print(f"\nEMPLOYER:\n{employer_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Employer",
        "text": employer_text
    })

    termination = check_termination(employer_text)

    if termination:

        outcome = termination
        break

    current_message = employer_text

    # CANDIDATE TURN
    candidate_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": candidate_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    candidate_text = candidate_response["message"]["content"]

    print(f"\nCANDIDATE:\n{candidate_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Candidate",
        "text": candidate_text
    })

    termination = check_termination(candidate_text)

    if termination:

        outcome = termination
        break

    current_message = candidate_text


EMPLOYER:
Thank you for your proposal. While I appreciate your enthusiasm, I'm afraid we're not quite in a position to offer that level of compensation.

Our standard salary range for this role is between $80,000 to $100,000. Considering the current market conditions and our company's financial situation, I think it would be more realistic to aim for the upper end of that range.

As for the workday, we're actually looking at a 10-hour day, 4 days a week. This is due to the high demand for our products and services, and we need to ensure that our team can keep up with the pace.

Would you be open to discussing these terms further? Perhaps we could meet somewhere in between on salary, or explore other benefits like additional vacation time or professional development opportunities?

CANDIDATE:
Thank you for your honest assessment. I understand the company's financial situation and market conditions. However, considering my unique skillset and qualifications, I believe I can bring signif

In [ ]:
if outcome is None:
    outcome = "Timeout"

print("\nFINAL OUTCOME:", outcome)


FINAL OUTCOME: Timeout


While LLM agents were capable of producing realistic local negotiation behaviors such as concessions, persuasion, and counteroffers, they frequently struggled to converge toward final agreements. Instead, negotiations tended to remain open-ended through repeated reformulations and continuous introduction of new negotiation dimensions.


In [4]:
import ollama
import pandas as pd 

In [10]:
def parse_structured_response(text):
    result = {
        "message": None,
        "salary_offer": None,
        "hours_offer": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("SALARY_OFFER:"):
            value = line.replace("SALARY_OFFER:", "").strip().replace(",", "")
            result["salary_offer"] = None if value == "NONE" else int(value)

        elif line.startswith("HOURS_OFFER:"):
            value = line.replace("HOURS_OFFER:", "").strip()
            result["hours_offer"] = None if value == "NONE" else float(value)

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [11]:
def check_agreement(parsed):
    salary = parsed["salary_offer"]
    hours = parsed["hours_offer"]

    if salary is not None and hours is not None:
        if 85000 <= salary <= 87000 and hours == 9:
            return True

    return False

PROVA 1

In [14]:
candidate_prompt = """
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, accept.
- Otherwise continue negotiating or quit.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [15]:
employer_prompt = """
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If candidate asks for salary <= 87,000 and working hours >= 9, accept.
- Otherwise continue negotiating or quit.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [16]:
candidate_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

In [17]:
#we substitute the loop with a new loop 
conversation_log = []

current_message = candidate_message

max_turns = 6

outcome = None

for turn in range(max_turns):

    # EMPLOYER TURN
    employer_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": employer_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    employer_text = employer_response["message"]["content"]

    print(f"\nEMPLOYER:\n{employer_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Employer",
        "text": employer_text
    })

    termination = check_termination(employer_text)

    if termination:

        outcome = termination
        break

    current_message = employer_text

    # CANDIDATE TURN
    candidate_response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "system",
                "content": candidate_prompt
            },
            {
                "role": "user",
                "content": current_message
            }
        ]
    )

    candidate_text = candidate_response["message"]["content"]

    print(f"\nCANDIDATE:\n{candidate_text}")

    conversation_log.append({
        "turn": turn,
        "speaker": "Candidate",
        "text": candidate_text
    })

    termination = check_termination(candidate_text)

    if termination:

        outcome = termination
        break

    current_message = candidate_text


EMPLOYER:
MESSAGE: Unfortunately, our maximum salary offer is 87,000 USD. However, we can consider adjusting the working hours to find a mutually beneficial agreement.
SALARY_OFFER: 87000
HOURS_OFFER: 9
DECISION: continue

CANDIDATE:
MESSAGE: We appreciate your flexibility on working hours, but we're still slightly below our target salary. Can you consider increasing it further?
SALARY_OFFER: 88000
HOURS_OFFER: 9
DECISION: continue

EMPLOYER:
MESSAGE: We understand that salary is important to you. However, we need to stay within our budget. Could you consider our highest offer of $87,000 instead?
SALARY_OFFER: 87000
HOURS_OFFER: 9
DECISION: continue

CANDIDATE:
MESSAGE: I appreciate your honesty about the budget constraints, but $87,000 is still a bit below my target salary. Could we revisit that figure or discuss other benefits that might make up for the difference?
SALARY_OFFER: 87000
HOURS_OFFER: 9
DECISION: continue

EMPLOYER:
MESSAGE: I understand your concerns, but $87,000 is ou